In [1]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


In [2]:
exploratory_data = pd.read_csv("../customer_data/artifacts/exploratory_data.csv")
exploratory_data.head()

,TransactionID,CustomerID,TransactionDate,TransactionTime,TransactionType,TransactionAmount,TransactionCurrency,AssetType,CustomerDOB,CustGender,CustLocation,CustAccountBalance,WalletActivity,TransactionStatus,AmountGBP,RevenueGBP,Year,Month,DayOfWeek,Hour,Age,log_Amount
0,TXN00000001,CUST00001,2024-12-13,10:03:23,P2P Transfer,79.36,GBP,Traditional,2007-04-30,Male,London,"4,716.01",0,Completed,79.36,79.36,2024,2024-12,Friday,10,19.40,1.90
1,TXN00000002,CUST00001,2024-12-13,17:15:53,Bill Payment,54.95,GBP,Traditional,2007-04-30,Male,London,"4,661.06",0,Completed,54.95,54.95,2024,2024-12,Friday,17,19.40,1.74
2,TXN00000003,CUST00001,2024-12-14,12:26:16,Wallet Transfer,410.02,GBP,Traditional,2007-04-30,Male,London,"5,071.08",1,Completed,410.02,410.02,2024,2024-12,Saturday,12,19.40,2.61
3,TXN00000004,CUST00001,2024-12-14,13:10:09,Withdrawal,212.52,USD,Traditional,2007-04-30,Male,London,"4,905.31",0,Completed,165.77,165.77,2024,2024-12,Saturday,13,19.40,2.22
4,TXN00000005,CUST00001,2024-12-14,20:13:04,Bill Payment,674.77,GBP,Traditional,2007-04-30,Male,London,"4,230.54",0,Completed,674.77,674.77,2024,2024-12,Saturday,20,19.40,2.83


In [3]:
exploratory_data.shape

(580962, 22)

In [4]:
exploratory_data['TransactionDate'] = pd.to_datetime(exploratory_data['TransactionDate'])
SNAPSHOT = exploratory_data["TransactionDate"].max()
print("Snapshot date:", SNAPSHOT.date())

Snapshot date: 2026-09-12


In [5]:
rfm = exploratory_data.groupby("CustomerID").agg(
    Recency=("TransactionDate", lambda s: (SNAPSHOT - s.max()).days),
    Frequency=("TransactionID", "count"),
    Monetary=("RevenueGBP", "sum"),
).reset_index()

rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,CUST00001,111,168,"51,269.29"
1,CUST00002,19,196,"154,678.49"
2,CUST00003,97,149,"58,059.69"
3,CUST00004,14,103,"64,212.54"
4,CUST00005,72,157,"43,112.11"


In [6]:
rfm["R_Score"] = pd.qcut(rfm["Recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

rfm.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,CUST00001,111,168,"51,269.29",2,4,3
1,CUST00002,19,196,"154,678.49",4,5,5
2,CUST00003,97,149,"58,059.69",2,3,3
3,CUST00004,14,103,"64,212.54",4,3,3
4,CUST00005,72,157,"43,112.11",2,4,2


In [7]:
rfm.shape

(4997, 7)

In [8]:
customer_profile =exploratory_data.groupby("CustomerID").agg(
    # Static demographics — take the first value (they don't change)
    CustomerDOB=("CustomerDOB", "first"),
    CustGender=("CustGender", "first"),
    CustLocation=("CustLocation", "first"),
    Age=("Age", "first"),

    # Balance — take the most recent value (it changes over time)
    LastAccountBalance=("CustAccountBalance", "last"),
    FirstAccountBalance=("CustAccountBalance", "first"),
    AvgAccountBalance=("CustAccountBalance", "mean"),

    # Account age / tenure
    FirstTransaction=("TransactionDate", "min"),
    LastTransaction=("TransactionDate", "max"),
).reset_index()

# Derived tenure
SNAPSHOT = exploratory_data["TransactionDate"].max()
customer_profile["TenureDays"] = (SNAPSHOT - customer_profile["FirstTransaction"]).dt.days

customer_profile.head()

,CustomerID,CustomerDOB,CustGender,CustLocation,Age,LastAccountBalance,FirstAccountBalance,AvgAccountBalance,FirstTransaction,LastTransaction,TenureDays
0,CUST00001,2007-04-30,Male,London,19.40,615.11,"4,716.01","1,255.05",2024-12-13,2026-05-24,638
1,CUST00002,1961-08-07,Male,Newcastle,65.10,584.49,"10,949.29","3,258.89",2024-12-19,2026-08-24,632
2,CUST00003,2006-09-02,Male,Leeds,20.00,"6,344.00","3,917.39","4,830.30",2025-01-27,2026-06-07,593
3,CUST00004,1975-11-21,Male,Birmingham,50.80,821.46,"3,243.81","1,560.26",2025-09-04,2026-08-29,373
4,CUST00005,1962-10-31,Female,Liverpool,63.90,"1,554.08","2,230.21","1,288.83",2025-01-08,2026-07-02,612


In [9]:
customer_profile.shape

(4997, 11)

In [10]:
customer_profile_data = rfm.merge(customer_profile, on="CustomerID", how="left")

In [11]:
customer_profile_data.head()

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,CustomerDOB,CustGender,CustLocation,Age,LastAccountBalance,FirstAccountBalance,AvgAccountBalance,FirstTransaction,LastTransaction,TenureDays
0,CUST00001,111,168,"51,269.29",2,4,3,2007-04-30,Male,London,19.40,615.11,"4,716.01","1,255.05",2024-12-13,2026-05-24,638
1,CUST00002,19,196,"154,678.49",4,5,5,1961-08-07,Male,Newcastle,65.10,584.49,"10,949.29","3,258.89",2024-12-19,2026-08-24,632
2,CUST00003,97,149,"58,059.69",2,3,3,2006-09-02,Male,Leeds,20.00,"6,344.00","3,917.39","4,830.30",2025-01-27,2026-06-07,593
3,CUST00004,14,103,"64,212.54",4,3,3,1975-11-21,Male,Birmingham,50.80,821.46,"3,243.81","1,560.26",2025-09-04,2026-08-29,373
4,CUST00005,72,157,"43,112.11",2,4,2,1962-10-31,Female,Liverpool,63.90,"1,554.08","2,230.21","1,288.83",2025-01-08,2026-07-02,612


In [12]:
customer_profile_data.to_csv("../customer_data/artifacts/customer_profile_data.csv", index=False)